<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/robertres.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install albumentations scikit-learn -q

import os
import cv2
import glob
import copy
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet34, ResNet34_Weights
import albumentations as A

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
import cv2
import glob
import copy
import random
import warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models import resnet34, ResNet34_Weights
import albumentations as A

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


BATCH_SIZE  = 8
EPOCHS      = 100
LR          = 1e-4
IMG_SIZE    = 256
SEED        = 42
N_SPLITS    = 5
TEST_SIZE   = 0.15   # 15% held-out test set
VAL_SIZE    = 0.15
CLASSES     = ["benign", "malignant"]
SAVE_DIR    = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True


train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(
        shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5,
        border_mode=cv2.BORDER_CONSTANT, value=0               # avoids artefacts on masks
    ),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),                                       # ultrasound speckle simulation
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

class BUSIDataset(Dataset):

    def __init__(self, base_dir, classes, transform=None, indices=None):
        self.transform = transform
        self._all_samples = []

        for cls in classes:
            cls_dir = os.path.join(base_dir, cls)
            if not os.path.exists(cls_dir):
                continue
            images = sorted([
                f for f in os.listdir(cls_dir)
                if f.endswith(".png") and "_mask" not in f
            ])
            label = 0 if cls == "benign" else 1
            for img_name in images:
                img_path  = os.path.join(cls_dir, img_name)
                base_name = img_name.replace(".png", "")
                mask_files = sorted([
                    f for f in os.listdir(cls_dir)
                    if f.startswith(base_name + "_mask") and f.endswith(".png")
                ])
                if not mask_files:
                    continue
                self._all_samples.append((
                    img_path,
                    [os.path.join(cls_dir, f) for f in mask_files],
                    label
                ))


        if indices is not None:
            self._all_samples = [self._all_samples[i] for i in indices]

    def __len__(self):
        return len(self._all_samples)

    def __getitem__(self, idx):
        img_path, mask_paths, label = self._all_samples[idx]

        image = np.array(Image.open(img_path).convert("RGB"))

        combined_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        for mpath in mask_paths:
            m = np.array(Image.open(mpath).convert("L"))
            combined_mask = np.logical_or(combined_mask, m > 0).astype(np.uint8)

        combined_mask = combined_mask.astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=combined_mask)
            image, combined_mask = aug["image"], aug["mask"]

        combined_mask = (combined_mask > 0.5).astype(np.float32)

        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float()
        mask_tensor  = torch.from_numpy(combined_mask).unsqueeze(0).float()
        return image_tensor, mask_tensor

    @property
    def labels(self):
        """Return label list for stratification."""
        return [s[2] for s in self._all_samples]



class RobertsEdgeOperator(nn.Module):
    """
    Fixed (non-trainable) Roberts Cross edge detector.
    Converts RGB to grayscale then computes diagonal gradient magnitude.
    Output: single-channel edge map, same spatial size as input.
    """
    def __init__(self):
        super().__init__()
        kx = torch.tensor([[[[1.0, 0.0], [0.0, -1.0]]]])
        ky = torch.tensor([[[[0.0, 1.0], [-1.0, 0.0]]]])
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

    def forward(self, x):
        gray = (0.2989 * x[:, 0:1]
               + 0.5870 * x[:, 1:2]
               + 0.1140 * x[:, 2:3])
        gray_padded = F.pad(gray, (0, 1, 0, 1), mode="replicate")
        gx   = F.conv2d(gray_padded, self.kx)
        gy   = F.conv2d(gray_padded, self.ky)
        edge = torch.sqrt(gx ** 2 + gy ** 2 + 1e-8)
        return edge


class PCBAM_Filter(nn.Module):
    """
    Parallel Channel-and-Spatial Attention Module.
    Applies channel attention (shared MLP on avg+max pool) then
    spatial attention (7×7 conv on avg+max channel maps).
    """
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.cam = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels)
        )
        self.sam = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()

        # Channel attention
        avg = self.cam(F.adaptive_avg_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        mxp = self.cam(F.adaptive_max_pool2d(x, 1).view(b, c)).view(b, c, 1, 1)
        x_c = x * torch.sigmoid(avg + mxp)

        # Spatial attention
        sp  = torch.cat([
            torch.mean(x_c, dim=1, keepdim=True),
            torch.max(x_c,  dim=1, keepdim=True)[0]
        ], dim=1)
        x_s = x_c * torch.sigmoid(self.sam(sp))
        return x_s


class SpatialGateAttention(nn.Module):
    """
    Spatial Gate Attention at the bottleneck.
    Lightweight self-attention: Q/K projections guide a gate over V features.
    Residual connection preserves gradient flow.
    """
    def __init__(self, in_channels):
        super().__init__()
        mid = in_channels // 8
        self.q_conv   = nn.Conv2d(in_channels, mid, 1)
        self.k_conv   = nn.Conv2d(in_channels, mid, 1)
        self.v_conv   = nn.Conv2d(in_channels, in_channels, 1)
        self.gate     = nn.Conv2d(mid * 2 + in_channels, in_channels, 1)

    def forward(self, x):
        q = self.q_conv(x)
        k = self.k_conv(x)
        v = self.v_conv(x)
        A = torch.sigmoid(self.gate(torch.cat([q, k, v], dim=1)))
        return x + v * A


class DoubleConv(nn.Module):
    """Standard double 3×3 conv block with BN and ReLU."""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class MaxDiceUNet(nn.Module):

    def __init__(self):
        super().__init__()

        # --- Edge operator (frozen, no parameters) ---
        self.roberts = RobertsEdgeOperator()

        # --- ResNet34 backbone ---
        resnet = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

        # Modify first conv to accept 4-channel input (RGB + edge)
        old_conv = resnet.conv1
        self.conv1 = nn.Conv2d(4, 64, kernel_size=7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.conv1.weight[:, :3] = old_conv.weight
            self.conv1.weight[:, 3]  = old_conv.weight.mean(dim=1)  # avg init for edge ch

        self.bn1     = resnet.bn1
        self.relu    = resnet.relu
        self.maxpool = resnet.maxpool

        # Encoder stages
        self.layer1 = resnet.layer1   # 64ch
        self.layer2 = resnet.layer2   # 128ch
        self.layer3 = resnet.layer3   # 256ch
        self.layer4 = resnet.layer4   # 512ch

        # PCBAM on skip connections
        self.pcbam1 = PCBAM_Filter(64)
        self.pcbam2 = PCBAM_Filter(128)
        self.pcbam3 = PCBAM_Filter(256)
        self.pcbam4 = PCBAM_Filter(512)

        # Bottleneck
        self.pool       = nn.MaxPool2d(2)        # 8×8 → 4×4
        self.bottleneck = DoubleConv(512, 1024)
        self.sga        = SpatialGateAttention(1024)

        # Decoder
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, stride=2)   # 4→8
        self.dec4 = DoubleConv(1024, 512)

        self.up3  = nn.ConvTranspose2d(512, 256, 2, stride=2)    # 8→16
        self.dec3 = DoubleConv(512, 256)

        self.up2  = nn.ConvTranspose2d(256, 128, 2, stride=2)    # 16→32
        self.dec2 = DoubleConv(256, 128)

        self.up1  = nn.ConvTranspose2d(128, 64, 2, stride=2)     # 32→64
        self.dec1 = DoubleConv(128, 64)

        # Extra stages: 64→128, 128→256 with x0 skip
        self.up0  = nn.ConvTranspose2d(64, 64, 2, stride=2)      # 64→128
        self.dec0 = DoubleConv(128, 64)                           # 64 + 64 skip

        self.up_out  = nn.ConvTranspose2d(64, 32, 2, stride=2)   # 128→256
        self.dec_out = DoubleConv(32, 32)
        self.final   = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        # Early fusion: concatenate edge map as 4th channel
        x_edge  = self.roberts(x)
        x_fused = torch.cat([x, x_edge], dim=1)       # B×4×256×256

        # Encoder
        x0 = self.relu(self.bn1(self.conv1(x_fused)))  # B×64×128×128
        x1 = self.maxpool(x0)                           # B×64×64×64

        e1 = self.layer1(x1)   # B×64×64×64
        e2 = self.layer2(e1)   # B×128×32×32
        e3 = self.layer3(e2)   # B×256×16×16
        e4 = self.layer4(e3)   # B×512×8×8

        # PCBAM-filtered skip connections
        s1 = self.pcbam1(e1)
        s2 = self.pcbam2(e2)
        s3 = self.pcbam3(e3)
        s4 = self.pcbam4(e4)

        # Bottleneck with spatial gate attention
        b = self.sga(self.bottleneck(self.pool(e4)))   # B×1024×4×4

        # Decoder with skip connections
        d4 = self.dec4(torch.cat([self.up4(b),  s4], dim=1))   # B×512×8×8
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))   # B×256×16×16
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))   # B×128×32×32
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))   # B×64×64×64

        d0  = self.dec0(torch.cat([self.up0(d1), x0], dim=1))  # B×64×128×128
        out = self.dec_out(self.up_out(d0))                     # B×32×256×256

        return self.final(out)                                  # B×1×256×256




class HybridLoss(nn.Module):
    """
    Weighted combination of:
      - Binary Cross-Entropy (handles class balance)
      - Dice Loss           (optimizes overlap directly)
      - Focal Loss          (down-weights easy negatives)
    Weights: BCE=0.4, Dice=0.4, Focal=0.2
    """
    def __init__(self, smooth=1e-5, focal_gamma=2.0):
        super().__init__()
        self.smooth      = smooth
        self.focal_gamma = focal_gamma

    def forward(self, logits, targets):
        bce   = F.binary_cross_entropy_with_logits(logits, targets)

        probs = torch.sigmoid(logits).view(-1)
        tgt   = targets.view(-1)

        inter = (probs * tgt).sum()
        dice  = 1.0 - (2.0 * inter + self.smooth) / (
                    probs.sum() + tgt.sum() + self.smooth)

        pt    = torch.where(tgt == 1, probs, 1.0 - probs)
        focal = (-(1.0 - pt) ** self.focal_gamma * torch.log(pt + 1e-8)).mean()

        return 0.4 * bce + 0.4 * dice + 0.2 * focal


def compute_metrics(logits, targets, threshold=0.5, smooth=1e-5):
    """
    Returns dict with Dice, IoU, Precision, Recall, F1
    from raw logits and binary target masks.
    """
    preds = (torch.sigmoid(logits) > threshold).float().view(-1)
    tgts  = targets.view(-1).float()

    tp = (preds * tgts).sum()
    fp = (preds * (1 - tgts)).sum()
    fn = ((1 - preds) * tgts).sum()

    precision = (tp + smooth) / (tp + fp + smooth)
    recall    = (tp + smooth) / (tp + fn + smooth)
    f1        = (2 * precision * recall) / (precision + recall + smooth)
    dice      = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    iou       = (tp + smooth) / (tp + fp + fn + smooth)

    return {
        "dice":      dice.item(),
        "iou":       iou.item(),
        "precision": precision.item(),
        "recall":    recall.item(),
        "f1":        f1.item()
    }


def aggregate_metrics(metric_list):
    """Average a list of metric dicts."""
    keys = metric_list[0].keys()
    return {k: float(np.mean([m[k] for m in metric_list])) for k in keys}





possible_paths = glob.glob("/kaggle/input/**/benign", recursive=True)
if not possible_paths:
    raise FileNotFoundError("Dataset not found. Check Kaggle input path.")
BASE_DIR     = os.path.dirname(possible_paths[0])
full_dataset = BUSIDataset(BASE_DIR, classes=CLASSES, transform=None)

all_indices  = np.arange(len(full_dataset))
all_labels   = np.array(full_dataset.labels)

print(f"Total samples : {len(all_indices)}")
print(f"  Benign      : {(all_labels == 0).sum()}")
print(f"  Malignant   : {(all_labels == 1).sum()}")


trainval_idx, test_idx = train_test_split(
    all_indices,
    test_size=TEST_SIZE,
    stratify=all_labels,
    random_state=SEED
)

trainval_labels = all_labels[trainval_idx]
test_labels     = all_labels[test_idx]

print(f"\nTrain+Val pool : {len(trainval_idx)}")
print(f"Test set (locked) : {len(test_idx)}")
print(f"  Test benign    : {(test_labels == 0).sum()}")
print(f"  Test malignant : {(test_labels == 1).sum()}")


skf       = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
criterion = HybridLoss()
fold_paths = []
fold_val_metrics = []

print(f"\n STARTING {N_SPLITS}-FOLD STRATIFIED CROSS VALIDATION")
print("=" * 60)

for fold, (rel_train_idx, rel_val_idx) in enumerate(
        skf.split(trainval_idx, trainval_labels)):


    abs_train_idx = trainval_idx[rel_train_idx]
    abs_val_idx   = trainval_idx[rel_val_idx]

    print(f"\n--- FOLD {fold+1}/{N_SPLITS} ---")
    print(f"  Train: {len(abs_train_idx)} | Val: {len(abs_val_idx)}")


    train_ds = BUSIDataset(BASE_DIR, CLASSES, train_transform,
                           indices=abs_train_idx)
    val_ds   = BUSIDataset(BASE_DIR, CLASSES, val_test_transform,
                           indices=abs_val_idx)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                              shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=2, pin_memory=True)

    model     = MaxDiceUNet().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_dice = 0.0
    best_weights  = None
    history       = {"train_dice": [], "val_dice": [], "val_iou": []}

    for epoch in range(EPOCHS):

        model.train()
        epoch_train_dice = []

        for images, masks in tqdm(train_loader,
                                  desc=f"F{fold+1} E{epoch+1:03d}",
                                  leave=False):
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                logits = model(images)
                loss   = criterion(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                m = compute_metrics(logits, masks)
                epoch_train_dice.append(m["dice"])

        scheduler.step()


        model.eval()
        val_metric_list = []

        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(device), masks.to(device)
                with torch.amp.autocast("cuda"):
                    logits = model(images)
                val_metric_list.append(compute_metrics(logits, masks))

        avg_train = float(np.mean(epoch_train_dice))
        avg_val   = aggregate_metrics(val_metric_list)

        history["train_dice"].append(avg_train)
        history["val_dice"].append(avg_val["dice"])
        history["val_iou"].append(avg_val["iou"])

        if avg_val["dice"] > best_val_dice:
            best_val_dice = avg_val["dice"]
            best_weights  = copy.deepcopy(model.state_dict())
            save_path     = os.path.join(SAVE_DIR, f"best_model_fold_{fold+1}.pth")
            torch.save(best_weights, save_path)

    fold_paths.append(save_path)
    fold_val_metrics.append(best_val_dice)

    print(f" Fold {fold+1} | Best Val Dice: {best_val_dice:.4f}")


    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history["train_dice"], label="Train Dice")
    ax.plot(history["val_dice"],   label="Val Dice")
    ax.plot(history["val_iou"],    label="Val IoU")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Score")
    ax.set_title(f"Fold {fold+1} Training Curve")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f"training_curve_fold_{fold+1}.png"), dpi=150)
    plt.close()

print("\n Cross-validation summary")
print("=" * 60)
for i, d in enumerate(fold_val_metrics):
    print(f"  Fold {i+1}: {d:.4f}")
print(f"  Mean ± Std: {np.mean(fold_val_metrics):.4f} ± {np.std(fold_val_metrics):.4f}")




def predict_tta(model, image_tensor):
    """
    Test-Time Augmentation with 3 views:
      1. Original
      2. Horizontal flip  (re-flipped before averaging)
      3. Vertical flip    (re-flipped before averaging)
    Returns averaged probability map (not binary).
    """
    with torch.amp.autocast("cuda"):
        p_orig  = torch.sigmoid(model(image_tensor))
        p_hflip = torch.sigmoid(model(torch.flip(image_tensor, dims=[3])))
        p_vflip = torch.sigmoid(model(torch.flip(image_tensor, dims=[2])))

    # Reverse the augmentations before averaging
    p_hflip = torch.flip(p_hflip, dims=[3])
    p_vflip = torch.flip(p_vflip, dims=[2])

    return (p_orig + p_hflip + p_vflip) / 3.0



test_ds     = BUSIDataset(BASE_DIR, CLASSES, val_test_transform,
                          indices=test_idx)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False,
                         num_workers=2, pin_memory=True)

# Load all fold models
ensemble = []
for path in fold_paths:
    m = MaxDiceUNet().to(device)
    m.load_state_dict(torch.load(path, weights_only=True))
    m.eval()
    ensemble.append(m)

print("\n FINAL EVALUATION ON HELD-OUT TEST SET")
print("=" * 60)


configs = {
    "Single model (fold 1), no TTA": (ensemble[:1], False),
    "Ensemble (5 models), no TTA":   (ensemble,     False),
    "Ensemble (5 models) + TTA":     (ensemble,     True),
}

results_table = {}

for config_name, (models, use_tta) in configs.items():
    all_metrics = []

    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc=config_name):
            images, masks = images.to(device), masks.to(device)

            # Average predictions across models
            avg_probs = torch.zeros_like(masks).to(device)
            for m in models:
                if use_tta:
                    avg_probs += predict_tta(m, images)
                else:
                    with torch.amp.autocast("cuda"):
                        avg_probs += torch.sigmoid(m(images))
            avg_probs /= len(models)


            pseudo_logit = torch.log(avg_probs.clamp(1e-6, 1 - 1e-6) /
                                     (1 - avg_probs.clamp(1e-6, 1 - 1e-6)))
            all_metrics.append(compute_metrics(pseudo_logit, masks))

    agg = aggregate_metrics(all_metrics)
    results_table[config_name] = agg

    print(f"\n  {config_name}")
    print(f"    Dice      : {agg['dice']:.4f}")
    print(f"    IoU       : {agg['iou']:.4f}")
    print(f"    Precision : {agg['precision']:.4f}")
    print(f"    Recall    : {agg['recall']:.4f}")
    print(f"    F1        : {agg['f1']:.4f}")

print("\n\n ABLATION TABLE")
print("=" * 80)
header = f"{'Configuration':<40} {'Dice':>6} {'IoU':>6} {'Prec':>6} {'Rec':>6} {'F1':>6}"
print(header)
print("-" * 80)
for name, m in results_table.items():
    print(f"{name:<40} {m['dice']:>6.4f} {m['iou']:>6.4f} "
          f"{m['precision']:>6.4f} {m['recall']:>6.4f} {m['f1']:>6.4f}")

print("\n Saving sample predictions...")

best_model = ensemble[np.argmax(fold_val_metrics)]

fig, axes = plt.subplots(4, 3, figsize=(10, 14))
axes[0, 0].set_title("Image",         fontsize=11)
axes[0, 1].set_title("Ground Truth",  fontsize=11)
axes[0, 2].set_title("Prediction",    fontsize=11)

mean_ = np.array([0.485, 0.456, 0.406])
std_  = np.array([0.229, 0.224, 0.225])

with torch.no_grad():
    sample_batch = next(iter(test_loader))
    imgs, msks = sample_batch
    imgs_gpu   = imgs.to(device)

    probs_list = [predict_tta(m, imgs_gpu) for m in ensemble]
    avg_p      = torch.stack(probs_list).mean(0)
    preds_bin  = (avg_p > 0.5).float().cpu().numpy()

for i in range(min(4, len(imgs))):
    img_np = imgs[i].permute(1, 2, 0).numpy()
    img_np = np.clip(img_np * std_ + mean_, 0, 1)

    axes[i, 0].imshow(img_np)
    axes[i, 0].axis("off")
    axes[i, 1].imshow(msks[i, 0].numpy(), cmap="gray")
    axes[i, 1].axis("off")
    axes[i, 2].imshow(preds_bin[i, 0], cmap="gray")
    axes[i, 2].axis("off")

plt.suptitle("MaxDiceUNet — Sample Predictions on Test Set", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "sample_predictions.png"),
            dpi=150, bbox_inches="tight")
plt.close()
print(f" Outputs saved to {SAVE_DIR}")

Using device: cuda
Using device: cuda
Total samples : 647
  Benign      : 437
  Malignant   : 210

Train+Val pool : 549
Test set (locked) : 98
  Test benign    : 66
  Test malignant : 32

 STARTING 5-FOLD STRATIFIED CROSS VALIDATION

--- FOLD 1/5 ---
  Train: 439 | Val: 110
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 201MB/s]


 Fold 1 | Best Val Dice: 0.8047

--- FOLD 2/5 ---
  Train: 439 | Val: 110


 Fold 2 | Best Val Dice: 0.7703

--- FOLD 3/5 ---
  Train: 439 | Val: 110


 Fold 3 | Best Val Dice: 0.8032

--- FOLD 4/5 ---
  Train: 439 | Val: 110


...

Ensemble (5 models), no TTA: 100%|██████████| 25/25 [00:02<00:00, 11.55it/s]



  Ensemble (5 models), no TTA
    Dice      : 0.8471
    IoU       : 0.7457
    Precision : 0.8794
    Recall    : 0.8298
    F1        : 0.8471


Ensemble (5 models) + TTA: 100%|██████████| 25/25 [00:05<00:00,  4.54it/s]


  Ensemble (5 models) + TTA
    Dice      : 0.8493
    IoU       : 0.7496
    Precision : 0.8815
    Recall    : 0.8323
    F1        : 0.8493


 ABLATION TABLE
Configuration                              Dice    IoU   Prec    Rec     F1
--------------------------------------------------------------------------------
Single model (fold 1), no TTA            0.7980 0.6799 0.7763 0.8348 0.7980
Ensemble (5 models), no TTA              0.8471 0.7457 0.8794 0.8298 0.8471
Ensemble (5 models) + TTA                0.8493 0.7496 0.8815 0.8323 0.8493

 Saving sample predictions...


 Outputs saved to /kaggle/working
